In [18]:
#!pip install thefuzz[speedup]
import numpy as np
from thefuzz import process
import pandas as pd

df=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")
Xmins_data=pd.read_csv("ML_training2.csv").iloc[:,1:]
df=df[["Full_Name", "value", "kickoff_time","team_code","news","selected"]]
df['kickoff_time'] = pd.to_datetime(df['kickoff_time'])
result = df.loc[df.groupby('Full_Name')['kickoff_time'].idxmax(), ['Full_Name', 'value', 'kickoff_time',"team_code","news","selected"]]

mins=Xmins_data.loc[Xmins_data.groupby('name')['time'].idxmax(), ['name', 'average_minutes']]
print(mins)

pred_df=pd.read_csv("All_Predictions.csv").iloc[:,1:]

# Create a dictionary for quick lookup
name_value_dict = dict(zip(result['Full_Name'], result['value']))

name_team_dict= dict(zip(result['Full_Name'], result['team_code']))

name_news_dict= dict(zip(result['Full_Name'], result['news']))

name_selected_dict= dict(zip(result['Full_Name'], result['selected']))
print(name_selected_dict)
# Function to find the best fuzzy match
def get_best_match(name, choices, threshold=80):
    # ExtractOne returns (match, score, index), so we take only the first two
    match_data = process.extractOne(name, choices)
    if match_data:
        match, score, _ = match_data
        if score >= threshold:
            return match
    return None

# Get the best match for each name in pred_df
pred_df['Matched_Name'] = pred_df['Name'].apply(lambda x: get_best_match(x, result['Full_Name']))

# Map the value from result to pred_df based on the best match
pred_df['value'] = pred_df['Matched_Name'].map(name_value_dict)/10
pred_df['team'] = pred_df['Matched_Name'].map(name_team_dict)
pred_df['news'] = pred_df['Matched_Name'].map(name_news_dict)
pred_df['selected'] = pred_df['Matched_Name'].map(name_selected_dict)

def process_news(text):
    if pd.isna(text) or text.strip() == "":  # Blank check
        return 1
    elif "%" in text:
        # Extract number before %
        import re
        match = re.search(r"(\d+)%", text)
        if match:
            return int(match.group(1))/100  # Return the number before %
    return 0  # Default to 0 if no %

# Apply function to create new column
pred_df['offset'] = pred_df['news'].apply(process_news)
#pred_df[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']] = pred_df[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']].mul(pred_df['offset'], axis=0)
pred_df = pd.merge(pred_df, mins, left_on='Name', right_on='name', how='left')

pred_df["selected"] = pred_df["selected"]/11000000
pred_df["minutes_multiplier"] = np.minimum(1, pred_df['average_minutes'] / 70)

    
#pred_df[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']] = pred_df[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']].mul(pred_df['minutes_multiplier'], axis=0)
pred_df['p0']=0
pred_df.to_csv("Optimize_players.csv")  

<>:6: SyntaxWarning: invalid escape sequence '\F'
<>:6: SyntaxWarning: invalid escape sequence '\F'
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_46596\3041741489.py:6: SyntaxWarning: invalid escape sequence '\F'
  df=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")


                     name  average_minutes
26103     Aaron_Cresswell            58.14
5038         Aaron_Hickey             0.00
23084     Aaron_Ramsdale0            89.97
23129     Aaron_Ramsdale1             0.22
32450        Aaron_Ramsey             0.07
...                   ...              ...
30372        Zeki_Amdouni            34.56
21678  Álex_Moreno Lopera             3.37
34180      Çaglar_Söyüncü            17.93
8781       Đorđe_Petrović             0.00
26207    Łukasz_Fabiański             0.03

[904 rows x 2 columns]
{'Aaron_Anselmino': 563, 'Aaron_Cresswell': 195939, 'Aaron_Hickey': 1308, 'Aaron_Ramsdale': 161121, 'Aaron_Wan-Bissaka': 604781, 'Abdoulaye_Doucouré': 73261, 'Abdukodir_Khusanov': 41188, 'Abdul_Fatawu': 19035, 'Adam_Armstrong': 57962, 'Adam_Lallana': 2998, 'Adam_Smith': 14836, 'Adam_Webster': 9219, 'Adam_Wharton': 16497, 'Adama_Traoré': 162438, 'Adrian_Mazilu': 1223, 'Albert_Grønbæk': 497, 'Alejandro_Garnacho': 554950, 'Alejo_Véliz': 567, 'Alex_Iwobi': 653

In [33]:
import pandas as pd

# Load data
df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
max_time = df["time"].max()

# Filter the data for the relevant time range
filter_df = df[(df["time"] < max_time - 6) & (df["time"] >= max_time - 26)]   

# Pivot the data so each player's points are in separate columns
df_pivot = filter_df.pivot(index="time", columns="name", values="total_points").fillna(2)

# Compute variance and mean for each player
variance_per_player = df_pivot.var()
mean_per_player = df_pivot.mean()

# Compute variance / mean²
stability_metric = variance_per_player / (mean_per_player ** 2)

# Create a DataFrame with 'name' and 'stability_metric' columns
stability_df = pd.DataFrame({
    "name": stability_metric.index,  
    "variance_over_mean_squared": stability_metric.values
})

# Save the results to a CSV file
stability_df = stability_df.sort_values(by="variance_over_mean_squared", ascending=False)
stability_df.to_csv("stability_metric.csv", index=False)

# Print the DataFrame
print(stability_df)



                          name  variance_over_mean_squared
348              Jack_Robinson                    3.906046
59            Anel_Ahmedhodžić                    3.673469
460        Junior_Firpo Adames                    3.262866
285  Gabriel_Fernando de Jesus                    3.078947
395               Jayden_Bogle                    3.054780
..                         ...                         ...
367           Jakub_Stolarczyk                    0.013149
146          Chiedozie_Ogbene0                    0.013149
501        Leander_Dendoncker1                    0.011898
857            Wayne_Hennessey                    0.000000
71              Antonín_Kinsky                    0.000000

[895 rows x 2 columns]


In [16]:

import requests
import requests
team_id=544468


def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{team_id}/transfers/"
    response_transfers = requests.get(transfers_url)

    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None

    transfers_data = response_transfers.json()
    return transfers_data

# Example Usage
team_transfers = get_transfers(team_id)

import pandas as pd
df = pd.DataFrame(team_transfers)

active=[]
for i in range(len(df["element_in"])):
    element_in=df["element_in"].values[-i-1]
    out_list=df["element_out"].iloc[0:-i-1].values
    if(element_in in out_list):
        active.append(0)
    else:
        active.append(1)
df["Active"]= list(reversed(active))

df=df[df["Active"]==1]
df=df[["element_in", "element_in_cost"]]

team_id = team_id  # Replace with your FPL team ID
gameweek = 28  # Replace with the desired gameweek

# API Endpoint
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/event/{gameweek}/picks/"

# Request Data
response = requests.get(url)

# Check if request is successful
if response.status_code == 200:
    team_selection = response.json()
    picks=team_selection.get("picks")  # View the JSON response
    pick_df = pd.DataFrame(picks)
    
else:
    print(f"Error fetching team selection (Status Code: {response.status_code})")
print(df)

for g in range(len(pick_df)):
    element=pick_df["element"].values[g]
    if(element in [109]):
        element=304
    if(element not in df["element_in"].values):
        new_row = pd.DataFrame({'element_in': [element], 'element_in_cost': [np.nan]}, index=[len(df)])
        df = pd.concat([df, new_row], ignore_index=True)
print(df)

data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")
data=data[["Full_Name","element", "value", "kickoff_time"]]
data['kickoff_time'] = pd.to_datetime(data['kickoff_time'])
result = data.loc[data.groupby('Full_Name')['kickoff_time'].idxmax(), ['Full_Name','element', 'value', 'kickoff_time']]

team_df=pd.merge(df, result, left_on='element_in', right_on='element', how='left')
team_df['element_in_cost'] = team_df['element_in_cost'].fillna(team_df['value'])
team_df["selling_price_value"] = np.floor((team_df["value"] - team_df["element_in_cost"]) / 2).clip(lower=0)
team_df["selling_price"] = (team_df[["value", "element_in_cost"]].min(axis=1)+team_df["selling_price_value"])/10
print(team_df)
pred_data=pd.read_csv("All_Predictions.csv").iloc[:,1:]["Name"]
team_df=team_df[team_df["element_in_cost"]>30]
new_Names=[]
name_list=pred_data.values
for j in range(len(team_df)):
    name=team_df["Full_Name"].values[j]
    if(name in name_list):
        new_Names.append(name)
    elif(name+'1' in name_list):
        new_Names.append(name+'1')
    elif(name+'0' in name_list):
        new_Names.append(name+'0')
        
team_df["Full_Name"]=new_Names     

team_df.to_csv("Squad_data.csv")



print(team_df)
team_id = team_id  # Replace with your FPL team ID
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    money_in_bank = data.get("last_deadline_bank", 0)/10  # Convert to actual value
else:
    print(f"Error fetching data (Status Code: {response.status_code})")



data = pd.read_csv("Optimize_players.csv")
players = data['Name'].tolist()
costs = data['value'].tolist()

squad=[]

for t in range (len(team_df)):
    name=team_df["Full_Name"].values[t]
    squad.append(players.index(name))
print(squad)

list1 = costs.copy()
list2 = team_df["selling_price"].values
# Update list1 with values from list2 at positions specified by indexes
for i in range(len(list2)):
    list1[squad[i]] = list2[i]  #

for j in range(len(list2)):
    print(list1[squad[j]])


<>:72: SyntaxWarning: invalid escape sequence '\F'
<>:72: SyntaxWarning: invalid escape sequence '\F'
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22648\2393417650.py:72: SyntaxWarning: invalid escape sequence '\F'
  data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")


    element_in  element_in_cost
0          447               72
1          533               47
2          741                8
3           74               51
5          235               50
6          580               44
8          231               43
10         211               47
11         366               84
12         110               62
13         311               70
14         401               85
16         328              131
20          78               56
22         182              109
    element_in  element_in_cost
0          447             72.0
1          533             47.0
2          741              8.0
3           74             51.0
4          235             50.0
5          580             44.0
6          231             43.0
7          211             47.0
8          366             84.0
9          110             62.0
10         311             70.0
11         401             85.0
12         328            131.0
13          78             56.0
14      

In [ ]:
WIldcard team

In [40]:
import pandas as pd
import numpy as np
from pulp import LpMaximize, LpProblem, LpVariable, lpSum

def wildcard_optimize_team(sel_thresh, budget, columns, file_path="Model_Optimizer.csv", current_player_path="Raw_Data_25/current_players.csv"):
    # Load and preprocess data
    data = pd.read_csv(file_path)
    current_players=pd.read_csv(current_player_path)
    for col in columns:
        data[col] = np.where(
            data["offset"] < 1,
            data[col] * data["offset"],
            data[col] * data["minutes_multiplier"]
        )

    # Setup inputs
    players = data['name'].tolist()
    positions = data['position'].tolist()
    costs = data['value'].tolist()
    teams = data['team_code'].tolist()
    selected = data['selected'].tolist()
    predicted_points = data[columns].values

    GW_range = len(columns)
    gameweeks = range(GW_range)
    num_players = len(players)

    # Model setup
    model = LpProblem("Maximize_Predicted_Points", LpMaximize)

    # Variables
    x = {(i, t): LpVariable(cat='Binary', name=f"x_{i}_{t}") for i in range(num_players) for t in gameweeks}
    y = {(i, t): LpVariable(cat='Binary', name=f"y_{i}_{t}") for i in range(num_players) for t in gameweeks}
    bench = {(i, t): LpVariable(cat='Binary', name=f"bench_{i}_{t}") for i in range(num_players) for t in gameweeks}
    bench_gk = {t: LpVariable(cat='Binary', name=f"bench_gk_{t}") for t in gameweeks}
    transfer_in = {(i, t): LpVariable(cat='Binary', name=f"transfer_in_{i}_{t}") for i in range(num_players) for t in range(1, 8)}
    transfer_out = {(i, t): LpVariable(cat='Binary', name=f"transfer_out_{i}_{t}") for i in range(num_players) for t in range(1, 8)}
    saved_transfers = {t: LpVariable(cat='Integer', lowBound=0, upBound=3, name=f"saved_transfers_{t}") for t in range(8)}

    # Objective
    model += lpSum(y[i, t] * predicted_points[i][t] for i in range(num_players) for t in gameweeks)

    # Constraints
    for t in gameweeks:
        model += lpSum(y[i, t] for i in range(num_players) if positions[i] == 'DEF') == 3
        for i in range(num_players):
            model += y[i, t] <= x[i, t]
            model += y[i, t] <= 1 - bench[i, t]
            model += y[i, t] >= x[i, t] + (1 - bench[i, t]) - 1
            model += x[i, t] * selected[i] <= sel_thresh

        model += lpSum(x[i, t] * costs[i] for i in range(num_players)) <= budget
        for team in set(teams):
            model += lpSum(x[i, t] for i in range(num_players) if teams[i] == team) <= 3
        model += lpSum(x[i, t] for i in range(num_players)) == 15
        model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'DEF') == 5
        model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'GKP') == 2
        model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'MID') == 5
        model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'FWD') == 3

        model += lpSum(bench[i, t] for i in range(num_players) if positions[i] == 'GKP') == 1
        model += lpSum(bench[i, t] for i in range(num_players) if positions[i] != 'GKP') == 3
        for i in range(num_players):
            model += bench[i, t] <= x[i, t]

    # Transfers
    for t in range(1, GW_range):
        for i in range(num_players):
            model += transfer_in[i, t] >= x[i, t] - x[i, t - 1]
            model += transfer_out[i, t] >= x[i, t - 1] - x[i, t]
            model += transfer_out[i, t] <= x[i, t - 1]
        model += lpSum(transfer_in[i, t] for i in range(num_players)) <= 1 + saved_transfers[t - 1]
        model += saved_transfers[t] == saved_transfers[t - 1] + (1 - lpSum(transfer_in[i, t] for i in range(num_players)))
        model += saved_transfers[t] <= 3

    model += saved_transfers[0] == 0

    # Solve
    model.solve()

    # Output
    print(f"Status: {model.status}")
    for t in gameweeks:
        print(f"\nGameweek {t+1} Squad:")
        for i in range(num_players):
            if x[i, t].varValue > 0.5:
                status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
                print(f"- {players[i]} ({positions[i]}) - {status}")

    for t in range(1, GW_range):
        print(f"\nTransfers for Gameweek {t+1}:")
        players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
        players_out = [players[i] for i in range(num_players) if transfer_out[i, t].varValue > 0.5]
        print(f"  In: {', '.join(players_in) if players_in else 'None'}")
        print(f"  Out: {', '.join(players_out) if players_out else 'None'}")



    records = []

    for t in gameweeks:
        for i in range(num_players):
            name = players[i]
            player_row_code=current_players[current_players["name"]==name]["code"].values[0]
            pos = positions[i]
            gw = t + 1  # Gameweek numbering

            if x[i, t].varValue > 0.5:
                if bench[i, t].varValue > 0.5:
                    status = "benched"
                else:
                    status = "playing"
                records.append({"Name": name, "status": status, "GW": gw, "position": pos, "photo":f"https://resources.premierleague.com/premierleague/photos/players/110x140/p{player_row_code}.png"})

    for t in range(1, GW_range):
        for i in range(num_players):
            name = players[i]
            player_row_code=current_players[current_players["name"]==name]["code"].values[0]
            pos = positions[i]
            gw = t + 1  # Transfers affect upcoming GW

            if transfer_in[i, t].varValue > 0.5:
                records.append({"Name": name, "status": "transferred_in", "GW": gw, "position": pos, "photo":f"https://resources.premierleague.com/premierleague/photos/players/110x140/p{player_row_code}.png"})
            if transfer_out[i, t].varValue > 0.5:
                records.append({"Name": name, "status": "transferred_out", "GW": gw, "position": pos, "photo":f"https://resources.premierleague.com/premierleague/photos/players/110x140/p{player_row_code}.png"})

    # Final structured DataFrame
    status_df = pd.DataFrame(records)
    status_df.to_csv("Wildcard_team.csv")
    print(status_df)

    
wildcard_optimize_team(0.9, 100, ['37', '38'])

Status: 1

Gameweek 1 Squad:
- Benjamin_White (DEF) - Playing
- Bryan_Mbeumo (MID) - Playing
- Daniel_Muñoz (DEF) - Playing
- David_Moyes (FWD) - Bench
- Dean_Huijsen (DEF) - Bench
- Erling_Haaland (FWD) - Playing
- Ismaïla_Sarr (MID) - Playing
- Jarrod_Bowen (MID) - Playing
- João_Neves Virgínia (GKP) - Playing
- Leandro_Trossard (MID) - Playing
- Milos_Kerkez (DEF) - Bench
- Mohamed_Salah (MID) - Playing
- Nick_Pope (GKP) - Bench
- Vitalii_Mykolenko (DEF) - Playing
- Yoane_Wissa (FWD) - Playing

Gameweek 2 Squad:
- Benjamin_White (DEF) - Playing
- Bryan_Mbeumo (MID) - Playing
- Daniel_Muñoz (DEF) - Bench
- David_Moyes (FWD) - Bench
- Dean_Huijsen (DEF) - Playing
- Erling_Haaland (FWD) - Playing
- Jarrod_Bowen (MID) - Playing
- João_Neves Virgínia (GKP) - Bench
- Justin_Kluivert (MID) - Playing
- Leandro_Trossard (MID) - Playing
- Milos_Kerkez (DEF) - Playing
- Mohamed_Salah (MID) - Playing
- Nick_Pope (GKP) - Playing
- Vitalii_Mykolenko (DEF) - Bench
- Yoane_Wissa (FWD) - Playing

Tr

In [ ]:
FREE HIT

In [34]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
import numpy as np

# Load Data
data = pd.read_csv("Model_Optimizer.csv")
cols = ["37"]
for col in cols:
    data[col] = np.where(data["offset"] < 1, data[col] * data["offset"], data[col] * data["minutes_multiplier"])
budget = 100.0  
sel_tresh = 0.9 # Max selected players
players = data['name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team_code'].tolist() 
selected = data['selected'].tolist() 
predicted_points = data[cols].values.flatten()  # Only use 'p1' for Gameweek 1

not_sel_list=[]

num_players = len(players)

# Define Model
model = LpProblem("Maximize_Predicted_Points_One_Round", LpMaximize)

# Decision Variables
x = {i: LpVariable(cat='Binary', name=f"x_{i}") for i in range(num_players)}  # Selected players
bench = {i: LpVariable(cat='Binary', name=f"bench_{i}") for i in range(num_players)}  # Bench players
y = {i: LpVariable(cat='Binary', name=f"y_{i}") for i in range(num_players)}  # Playing players

# Objective: Maximize Total Points for Gameweek 1
model += lpSum(y[i] * predicted_points[i] for i in range(num_players))

# Ensure y is 1 only when player is in the squad and not benched
for i in range(num_players):
    model += y[i] <= x[i]               # Can only play if selected in squad
    model += y[i] <= 1 - bench[i]        # Can't play if benched
    model += y[i] >= x[i] + (1 - bench[i]) - 1  # Consistency


for i in range(num_players):
        model += x[i] * selected[i] <= sel_tresh


model += lpSum(y[i] for i in range(num_players) if positions[i] == 'DEF') == 3

# Budget Constraint
model += lpSum(x[i] * costs[i] for i in range(num_players)) <= budget

# Max 3 Players per Team Constraint
for team in set(teams):
    model += lpSum(x[i] for i in range(num_players) if teams[i] == team) <= 3

# Total Players Constraint (15 players in squad)
model += lpSum(x[i] for i in range(num_players)) == 15

# Position Constraints
model += lpSum(x[i] for i in range(num_players) if positions[i] == 'DEF') == 5  # 5 Defenders
model += lpSum(x[i] for i in range(num_players) if positions[i] == 'GK') == 2   # 2 Goalkeepers
model += lpSum(x[i] for i in range(num_players) if positions[i] == 'MID') == 5  # 5 Midfielders
model += lpSum(x[i] for i in range(num_players) if positions[i] == 'FWD') == 3  # 3 Attackers

# Bench Constraints
model += lpSum(bench[i] for i in range(num_players) if positions[i] == 'GK') == 1  # Exactly 1 GK on bench
model += lpSum(bench[i] for i in range(num_players) if positions[i] != 'GK') == 3  # Exactly 3 outfield players on bench

# A player can only be benched if they are in the squad
for i in range(num_players):
    model += bench[i] <= x[i]

# Solve the Model
model.solve()

# Check the status of the solution
print(f"Status: {model.status}")

# Display selected players
result_set=[]
print("\nGameweek 1 Squad:")
current_players=pd.read_csv("Raw_Data_25/current_players.csv")
for i in range(num_players):
    player_set=[]
    if x[i].varValue > 0.5:
        try:
            player_row=current_players[current_players["name"]==players[i]]["code"].values[0]
        except:
            player_row=current_players[current_players["name"]==players[i][:-1]]["code"].values[0]
        status = "Bench" if bench[i].varValue > 0.5 else "Playing"
        print(f"- {players[i]} ({positions[i]}) - {status}")
        player_set.append(players[i])
        player_set.append(positions[i])
        player_set.append(status)
        player_set.append(f"https://resources.premierleague.com/premierleague/photos/players/110x140/p{player_row}.png")
        result_set.append(player_set)
columns=["Name", "position", "status","photo"]

result_df=pd.DataFrame(result_set,columns=columns)
result_df.to_csv("Free_hit_team.csv")
print(result_df)


Status: 1

Gameweek 1 Squad:
- Ashley_Young (DEF) - Playing
- Bryan_Mbeumo (MID) - Playing
- Cameron_Burgess (DEF) - Bench
- Daniel_Iversen (GK) - Bench
- Daniel_Muñoz (DEF) - Playing
- Eberechi_Eze (MID) - Playing
- Erling_Haaland (FWD) - Playing
- Ismaïla_Sarr (MID) - Playing
- Massimo_Luongo (MID) - Bench
- Mohamed_Salah (MID) - Playing
- Omar_Marmoush (FWD) - Playing
- Vitalii_Mykolenko (DEF) - Playing
- Wayne_Hennessey (GK) - Playing
- Yerson_Mosquera (DEF) - Bench
- Yoane_Wissa (FWD) - Playing
                 Name position   status  \
0        Ashley_Young      DEF  Playing   
1        Bryan_Mbeumo      MID  Playing   
2     Cameron_Burgess      DEF    Bench   
3      Daniel_Iversen       GK    Bench   
4        Daniel_Muñoz      DEF  Playing   
5        Eberechi_Eze      MID  Playing   
6      Erling_Haaland      FWD  Playing   
7        Ismaïla_Sarr      MID  Playing   
8      Massimo_Luongo      MID    Bench   
9       Mohamed_Salah      MID  Playing   
10      Omar_Marmoush 

In [55]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD

# Load Data
data = pd.read_csv("Optimize_players.csv")

budget = 103.0  
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist() 
predicted_points = data[['p0','p1', 'p2', 'p3','p4','p5','p6','p7','p8']].values


initial_squad=[players.index('Cody_Gakpo'),players.index('Alexander_Isak'),players.index('Yoane_Wissa'),players.index('Antoine_Semenyo'),
               players.index('Mohamed_Salah'),players.index('Dango_Ouattara'),players.index('Bruno_Borges Fernandes'),players.index('Cole_Palmer0'),
               players.index('Trent_Alexander-Arnold'),players.index('Daniel_Muñoz'),players.index('Dean_Huijsen'),players.index('Vitalii_Mykolenko'),
               players.index('Jacob_Greaves'),players.index('Jordan_Pickford'),players.index('Łukasz_Fabiański')]

#optimize_range=5
optimize_range=9
gameweeks = range(optimize_range)
num_players = len(players)

# Define Model
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables
x = {(i, t): LpVariable(cat='Binary', name=f"x_{i}_{t}") for i in range(num_players) for t in gameweeks}

for i in range(num_players):
    if i in initial_squad:
        model += x[i, 0] == 1
    else:
        model += x[i, 0] == 0

# Bench Variables
bench = {(i, t): LpVariable(cat='Binary', name=f"bench_{i}_{t}") for i in range(num_players) for t in gameweeks}
bench_gk = {t: LpVariable(cat='Binary', name=f"bench_gk_{t}") for t in gameweeks}

# Transfer Tracking Variables
transfer_in = {(i, t): LpVariable(cat='Binary', name=f"transfer_in_{i}_{t}") for i in range(num_players) for t in range(0, optimize_range)}
transfer_out = {(i, t): LpVariable(cat='Binary', name=f"transfer_out_{i}_{t}") for i in range(num_players) for t in range(0, optimize_range)}
saved_transfers = {t: LpVariable(cat='Integer', lowBound=0, upBound=3, name=f"saved_transfers_{t}") for t in range(optimize_range)}

c = {(i, t): LpVariable(cat='Binary', name=f"captain_{i}_{t}") for i in range(num_players) for t in gameweeks}

        
# Decision Variable for Playing Status
y = {(i, t): LpVariable(cat='Binary', name=f"y_{i}_{t}") for i in range(num_players) for t in gameweeks}

for t in gameweeks:
    model += lpSum(y[i, t] for i in range(num_players) if positions[i] == 'DEF') == 3

# Objective: Maximize Total Points (only for playing players)
model += lpSum((y[i, t] + c[i, t]) * predicted_points[i][t] for i in range(num_players) for t in range(1, optimize_range))

for t in range(0, optimize_range):
    model += lpSum(transfer_in[i, t] for i in range(num_players)) == lpSum(transfer_out[i, t] for i in range(num_players))
for t in range(1, optimize_range):
    for i in range(num_players):
        model += x[i, t] >= x[i, t - 1] - transfer_out[i, t]  # If not transferred out, stays in squad
        
# Ensure y is 1 only when player is in the squad and not benched
for t in gameweeks:
    for i in range(num_players):
        model += y[i, t] <= x[i, t]                  # Can only play if in squad
        model += y[i, t] <= 1 - bench[i, t]           # Can't play if benched
        model += y[i, t] >= x[i, t] + (1 - bench[i, t]) - 1  # Consistency

#captancy
for t in gameweeks:
    model += lpSum(c[i, t] for i in range(num_players)) == 1  # Only one captain per GW

for t in gameweeks:
    for i in range(num_players):
        model += c[i, t] <= y[i, t]  # Captain must be a playing player
        
    
# Budget Constraint
for t in gameweeks:
    model += lpSum(x[i, t] * costs[i] for i in range(num_players)) <= budget

# Max 3 Players per Team Constraint
for t in gameweeks:
    for team in set(teams):
        model += lpSum(x[i, t] for i in range(num_players) if teams[i] == team) <= 3

# Total Players Constraint
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players)) == 15


# Position Constraints for each Gameweek
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'DEF') == 5  # 5 Defenders
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'GK') == 2   # 2 Goalkeepers
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'MID') == 5  # 5 Midfielders
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'FWD') == 3  # 3 Attackers

# Bench Constraints
for t in gameweeks:
    # Exactly 1 goalkeeper on the bench
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] == 'GK') == 1
    # Exactly 3 outfield players on the bench
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] != 'GK') == 3
    
    for i in range(num_players):
        # A player can only be benched if they are in the squad
        model += bench[i, t] <= x[i, t]

# Transfer Constraints
for t in gameweeks[1:]:
    for i in range(num_players):
        model += transfer_in[i, t] >= x[i, t] - x[i, t - 1]
        model += transfer_out[i, t] >= x[i, t - 1] - x[i, t]
        model += transfer_out[i, t] <= x[i, t - 1]

    # Number of transfers allowed per week (considering saved transfers)
    model += lpSum(transfer_in[i, t] for i in range(num_players)) <= 1 + saved_transfers[t - 1]

    # Define saved transfers: If 1 or 0 transfers used, they are saved for the next week (capped at 2)
    model += saved_transfers[t] == saved_transfers[t - 1] + (1 - lpSum(transfer_in[i, t] for i in range(num_players)))
    model += saved_transfers[t] <= 3  

# Initial Transfers (Gameweek 1 starts with 1 available transfer)
model += saved_transfers[0] == 0  

# Solve the Model
model.solve()

# Check the status of the solution
print(f"Status: {model.status}")

# Display selected players for each gameweek
for t in range(1, optimize_range):
    print(f"\nGameweek {t+27} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

# Display Transfers for Each Gameweek
for t in range(1, optimize_range):  
    print(f"\nTransfers for Gameweek {t+27}:")
    players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
    players_out = [players[i] for i in range(num_players) if transfer_out[i, t].varValue > 0.5]

    if players_in or players_out:
        print(f"  In: {', '.join(players_in) if players_in else 'None'}")
        print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
    else:
        print("  No transfers this week.")


for t in range(1, optimize_range):
    for i in range(num_players):
        if c[i, t].varValue > 0.5:
            print(f"Gameweek {t} Captain: {players[i]}")

Status: 1

Gameweek 28 Squad:
- Yoane_Wissa (FWD) - Playing
- Cody_Gakpo (FWD) - Bench
- Alexander_Isak (FWD) - Playing
- Dean_Huijsen (DEF) - Playing
- Daniel_Muñoz (DEF) - Playing
- Vitalii_Mykolenko (DEF) - Bench
- Jacob_Greaves (DEF) - Bench
- Trent_Alexander-Arnold (DEF) - Playing
- Justin_Kluivert (MID) - Playing
- Dango_Ouattara (MID) - Playing
- Cole_Palmer0 (MID) - Playing
- Mohamed_Salah (MID) - Playing
- Bruno_Borges Fernandes (MID) - Playing
- Jordan_Pickford (GK) - Playing
- Łukasz_Fabiański (GK) - Bench

Gameweek 29 Squad:
- Yoane_Wissa (FWD) - Playing
- Alexander_Isak (FWD) - Playing
- Matheus_Santos Carneiro Da Cunha (FWD) - Playing
- Dean_Huijsen (DEF) - Playing
- Daniel_Muñoz (DEF) - Bench
- Vitalii_Mykolenko (DEF) - Playing
- Jacob_Greaves (DEF) - Playing
- Trent_Alexander-Arnold (DEF) - Bench
- Justin_Kluivert (MID) - Playing
- Dango_Ouattara (MID) - Playing
- Cole_Palmer0 (MID) - Playing
- Mohamed_Salah (MID) - Bench
- Bruno_Borges Fernandes (MID) - Playing
- Jorda

In [ ]:
Team_Optimize

In [25]:
import requests

def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{182285}/transfers/"
    response_transfers = requests.get(transfers_url)

    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None

    transfers_data = response_transfers.json()
    return transfers_data

# Example Usage
team_transfers = get_transfers(team_id)

import pandas as pd
df = pd.DataFrame(team_transfers)
print(df)


transfers1 = df.groupby('event').size().reset_index(name='count')
max_event = 28
print("Max event:", max_event)
saved_transfers = 0
last_event = 0

for h in range(max_event):
    new_event = last_event + 1
    if new_event in transfers1["event"].values:
        ind = transfers1["event"].tolist().index(new_event)
        transfers_made = transfers1["count"].values[ind]
        saved_transfers = saved_transfers - transfers_made
        saved_transfers = max(0, saved_transfers)
    else:
        saved_transfers += 1
    last_event = new_event
saved_transfers+=1
print("Saved transfers:", saved_transfers)





    element_in  element_in_cost  element_out  element_out_cost   entry  event  \
0          741                8          737                 8  182285     26   
1           74               51          327                75  182285     26   
2           12               46           54                53  182285     26   
3          737                8          742                 5  182285     25   
4           99               78          398                75  182285     24   
5          211               48          355                45  182285     23   
6          447               69          180                80  182285     22   
7          398               73          366                84  182285     19   
8          473               44          447                63  182285     18   
9          311               71          533                48  182285     18   
10         327               75           99                74  182285     18   
11         180              

In [ ]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import requests
import numpy as np

#eliot-239743, Aria-182285
team_id=544468
wildcard_round = 3  # Gameweek 3 (Index t=2)
bench_points_gw=20
Last_GW=28
initial_saved=1


def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{team_id}/transfers/"
    response_transfers = requests.get(transfers_url)

    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None

    transfers_data = response_transfers.json()
    return transfers_data

# Example Usage
team_transfers = get_transfers(team_id)

import pandas as pd
df = pd.DataFrame(team_transfers)

active=[]
for i in range(len(df["element_in"])):
    element_in=df["element_in"].values[-i-1]
    out_list=df["element_out"].iloc[0:-i-1].values
    if(element_in in out_list):
        active.append(0)
    else:
        active.append(1)
df["Active"]= list(reversed(active))

df=df[df["Active"]==1]
df=df[["element_in", "element_in_cost"]]

team_id = team_id  # Replace with your FPL team ID
gameweek = Last_GW  # Replace with the desired gameweek

# API Endpoint
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/event/{gameweek}/picks/"

# Request Data
response = requests.get(url)

# Check if request is successful
if response.status_code == 200:
    team_selection = response.json()
    picks=team_selection.get("picks")  # View the JSON response
    pick_df = pd.DataFrame(picks)
    
else:
    print(f"Error fetching team selection (Status Code: {response.status_code})")
print(df)

for g in range(len(pick_df)):
    element=pick_df["element"].values[g]
    if(element in [109]):
        element=304
    if(element not in df["element_in"].values):
        new_row = pd.DataFrame({'element_in': [element], 'element_in_cost': [np.nan]}, index=[len(df)])
        df = pd.concat([df, new_row], ignore_index=True)
print(df)

data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")
data=data[["Full_Name","element", "value", "kickoff_time"]]
data['kickoff_time'] = pd.to_datetime(data['kickoff_time'])
result = data.loc[data.groupby('Full_Name')['kickoff_time'].idxmax(), ['Full_Name','element', 'value', 'kickoff_time']]

team_df=pd.merge(df, result, left_on='element_in', right_on='element', how='left')
team_df['element_in_cost'] = team_df['element_in_cost'].fillna(team_df['value'])
team_df["selling_price_value"] = np.floor((team_df["value"] - team_df["element_in_cost"]) / 2).clip(lower=0)
team_df["selling_price"] = (team_df[["value", "element_in_cost"]].min(axis=1)+team_df["selling_price_value"])/10
print(team_df)
pred_data=pd.read_csv("All_Predictions.csv").iloc[:,1:]["Name"]
team_df=team_df[team_df["element_in_cost"]>30]
new_Names=[]
name_list=pred_data.values
for j in range(len(team_df)):
    name=team_df["Full_Name"].values[j]
    if(name in name_list):
        new_Names.append(name)
    elif(name+'1' in name_list):
        new_Names.append(name+'1')
    elif(name+'0' in name_list):
        new_Names.append(name+'0')
        
team_df["Full_Name"]=new_Names     

team_df.to_csv("Squad_data.csv")


# Load Data
data = pd.read_csv("Optimize_players.csv")
squad=pd.read_csv("Squad_data.csv")

url = f"https://fantasy.premierleague.com/api/entry/{team_id}/"
response = requests.get(url)
if response.status_code == 200:
    resonsep_data = response.json()
    
else:
    print(f"Error fetching data (Status Code: {response.status_code})")

money_in_bank_init = resonsep_data.get("last_deadline_bank", 0)/10  # Convert to actual value


players = data['Name'].tolist()
costs = data['value'].tolist()
initial_squad=[]
for t in range (len(squad)):
    name=squad["Full_Name"].values[t]
    initial_squad.append(players.index(name))


list1 = costs.copy()
selling_cost = squad["selling_price"].values

budget_amount=sum(selling_cost)+money_in_bank_init
print(budget_amount)
# Update list1 with values from list2 at positions specified by indexes
for i in range(len(selling_cost)):
    list1[initial_squad[i]] = selling_cost[i]  

# Define Constants
#budget = 103.0  
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist()
predicted_points = data[['p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8']].values
# Initial Squad
"""initial_squad=[players.index('Chris_Wood0'),players.index('Alexander_Isak'),players.index('Yoane_Wissa'),players.index('Antoine_Semenyo'),
               players.index('Mohamed_Salah'),players.index('Dango_Ouattara'),players.index('Bruno_Borges Fernandes'),players.index('Cole_Palmer0'),
               players.index('Trent_Alexander-Arnold'),players.index('Daniel_Muñoz'),players.index('Dean_Huijsen'),players.index('Vitalii_Mykolenko'),
               players.index('Rayan_Aït-Nouri'),players.index('Jordan_Pickford'),players.index('Łukasz_Fabiański')]"""


# Define Gameweeks
optimize_range = 9  # Number of gameweeks to optimize
gameweeks = range(optimize_range)
num_players = len(players)

# Define Wildcard Round (where unlimited transfers are allowed)

# Define Model
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables
x = {(i, t): LpVariable(cat='Binary', name=f"x_{i}_{t}") for i in range(num_players) for t in gameweeks}
bench = {(i, t): LpVariable(cat='Binary', name=f"bench_{i}_{t}") for i in range(num_players) for t in gameweeks}
c = {(i, t): LpVariable(cat='Binary', name=f"captain_{i}_{t}") for i in range(num_players) for t in gameweeks}
y = {(i, t): LpVariable(cat='Binary', name=f"y_{i}_{t}") for i in range(num_players) for t in gameweeks}
transfer_in = {(i, t): LpVariable(cat='Binary', name=f"transfer_in_{i}_{t}") for i in range(num_players) for t in gameweeks}
transfer_out = {(i, t): LpVariable(cat='Binary', name=f"transfer_out_{i}_{t}") for i in range(num_players) for t in gameweeks}
saved_transfers = {t: LpVariable(cat='Integer', lowBound=0, upBound=3, name=f"saved_transfers_{t}") for t in gameweeks}
money_in_bank_var = {t: LpVariable(f"money_in_bank_{t}", lowBound=0, cat='Continuous') for t in gameweeks}

# Initial Squad Constraint (Gameweek 1)
for i in range(num_players):
    model += x[i, 0] == (1 if i in initial_squad else 0)

# Objective: Maximize Total Points (only for playing players)
#model += lpSum((y[i, t] + c[i, t]) * predicted_points[i][t] for i in range(num_players) for t in gameweeks)

model += lpSum(
    (y[i, t] + c[i, t]) * predicted_points[i][t] + 
    (bench[i, t] * predicted_points[i][t] if t == bench_points_gw else 0)
    for i in range(num_players) for t in gameweeks
)
# Position Constraints
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players)) == 15  # Squad size 15
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'DEF') == 5
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'GK') == 2
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'MID') == 5
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'FWD') == 3

# Bench Constraints (1 GK, 3 Outfield)
for t in gameweeks:
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] == 'GK') == 1
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] != 'GK') == 3
    for i in range(num_players):
        model += bench[i, t] <= x[i, t]


for t in gameweeks:
    model += lpSum(y[i, t] for i in range(num_players) if positions[i] == 'DEF') >= 3  # Ensure at least 3 defenders play

# Ensure playing status
for t in gameweeks:
    for i in range(num_players):
        model += y[i, t] <= x[i, t]
        model += y[i, t] <= 1 - bench[i, t]
        model += y[i, t] >= x[i, t] + (1 - bench[i, t]) - 1

# Captain Selection
for t in gameweeks:
    model += lpSum(c[i, t] for i in range(num_players)) == 1
    for i in range(num_players):
        model += c[i, t] <= y[i, t]

#Budget

for t in gameweeks[1:]:
        model += money_in_bank_var[t] == money_in_bank_var[t-1] + lpSum(transfer_out[i, t] * list1[i] for i in range(num_players)) - lpSum(transfer_in[i, t] * costs[i] for i in range(num_players))

    #model += money_in_bank_var[t] == money_in_bank_var[t - 1]+(lpSum(transfer_out[i, t]*list1[i] for i in range(num_players))) -(lpSum(transfer_in[i, t]*costs[i] for i in range(num_players)))
for t in gameweeks:
    model += lpSum(x[i, t] * list1[i] for i in range(num_players)) + money_in_bank_var[t] == budget_amount


# Max 3 Players per Team
for t in gameweeks:
    for team in set(teams):
        model += lpSum(x[i, t] for i in range(num_players) if teams[i] == team) <= 3

# Transfer Constraints
for t in gameweeks[1:]:
    if t == wildcard_round:
        # Wildcard round: No transfer limits
        for i in range(num_players):
            model += x[i, t] >= x[i, t - 1] - transfer_out[i, t]
            model += x[i, t] <= x[i, t - 1] + transfer_in[i, t]
    else:
        # Normal transfer constraints
        for i in range(num_players):
            model += transfer_in[i, t] >= x[i, t] - x[i, t - 1]
            model += transfer_out[i, t] >= x[i, t - 1] - x[i, t]
            model += transfer_out[i, t] <= x[i, t - 1]

        model += lpSum(transfer_in[i, t] for i in range(num_players)) <= 1 + saved_transfers[t - 1]
for t in gameweeks[1:]:
    if t == wildcard_round:
        model += saved_transfers[t] == 0  # Reset saved transfers after wildcard
    else:
        model += saved_transfers[t] == saved_transfers[t - 1] + (1 - lpSum(transfer_in[i, t] for i in range(num_players)))
        model += saved_transfers[t] <= 3  # Maximum of 3 saved transfers
        

# Initial Transfers
model += saved_transfers[0] == initial_saved 
model += money_in_bank_var[0] == money_in_bank_init

# Solve Model
model.solve()
#model.solve(PULP_CBC_CMD(msg=True))
# Display Results
print(f"Status: {model.status}")
print(f"\nTotal Optimized Predicted Points: {model.objective.value()}")
# Display Squad & Transfers
for t in range(1, optimize_range):
    print(f"\nGameweek {t+28} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

# Display Transfers for Each Gameweek
for t in range(1, optimize_range):  
    print(f"\nTransfers for Gameweek {t+28}:")
    if t == wildcard_round:
        print("  Wildcard Activated! All players can be changed.")
    else:
        players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
        players_out = [players[i] for i in range(num_players) if x[i, t-1].varValue > 0.5 and x[i, t].varValue < 0.5]

        if players_in or players_out:
            print(f"  In: {', '.join(players_in) if players_in else 'None'}")
            print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
        else:
            print("  No transfers this week.")
# Display Captains
for t in range(1, optimize_range):
    for i in range(num_players):
        if c[i, t].varValue > 0.5:
            print(f"Gameweek {t+28} Captain: {players[i]}")

#569.8

<>:72: SyntaxWarning: invalid escape sequence '\F'
<>:72: SyntaxWarning: invalid escape sequence '\F'
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_20060\2570932982.py:72: SyntaxWarning: invalid escape sequence '\F'
  data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")


    element_in  element_in_cost
0          447               72
1          533               47
2          741                8
3           74               51
5          235               50
6          580               44
8          231               43
10         211               47
11         366               84
12         110               62
13         311               70
14         401               85
16         328              131
20          78               56
22         182              109
    element_in  element_in_cost
0          447             72.0
1          533             47.0
2          741              8.0
3           74             51.0
4          235             50.0
5          580             44.0
6          231             43.0
7          211             47.0
8          366             84.0
9          110             62.0
10         311             70.0
11         401             85.0
12         328            131.0
13          78             56.0
14      

In [22]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
from pulp import HiGHS_CMD
import requests
import numpy as np

#eliot-239743, Aria-182285, meg-544468
team_id=544468

wildcard_round = 10  # Gameweek 3 (Index t=2)
bench_points_gw=10
Last_GW=37
hit=1

banned_list=[]


def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{team_id}/transfers/"
    response_transfers = requests.get(transfers_url)

    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None

    transfers_data = response_transfers.json()
    return transfers_data

# Example Usage
team_transfers = get_transfers(team_id)

import pandas as pd
df = pd.DataFrame(team_transfers)

#Find if free hit used
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/history/"
resp = requests.get(url)
if resp.status_code == 200:
    chip_data = resp.json()
    chips = chip_data.get("chips", [])
    print(chips)
    # find the Free Hit chip
    freehit = next((c for c in chips if c.get("name") == "freehit"), None)
    if freehit:
        free_hit_gw_played = freehit.get("event")
    else:
        free_hit_gw_played=39
print(free_hit_gw_played)
print(df)
df=df[df["event"]!=free_hit_gw_played]


transfers1 = df.groupby('event').size().reset_index(name='count')

max_event = Last_GW
print("Max event:", max_event)
saved_transfers = 0
last_event = 0

for h in range(max_event):
    new_event = last_event + 1
    if new_event in transfers1["event"].values:
        ind = transfers1["event"].tolist().index(new_event)
        transfers_made = transfers1["count"].values[ind]
        saved_transfers = saved_transfers - transfers_made
        saved_transfers = max(0, saved_transfers)
    else:
        saved_transfers += 1
    last_event = new_event
initial_saved=saved_transfers+hit
print("Available Transfers")
print(initial_saved)

active=[]
for i in range(len(df["element_in"])):
    element_in=df["element_in"].values[-i-1]
    out_list=df["element_out"].iloc[0:-i-1].values
    if(element_in in out_list):
        active.append(0)
    else:
        active.append(1)
df["Active"]= list(reversed(active))
print(df)
df=df[df["Active"]==1]

df.to_csv("debug_team.csv")
df=df[["element_in", "element_in_cost"]]

team_id = team_id  # Replace with your FPL team ID
if(Last_GW==free_hit_gw_played):
    gameweek = Last_GW-1
    initial_saved-=1
else:
    gameweek = Last_GW  # Replace with the desired gameweek

# API Endpoint
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/event/{gameweek}/picks/"

# Request Data
response = requests.get(url)

# Check if request is successful
if response.status_code == 200:
    team_selection = response.json()
    picks=team_selection.get("picks")  # View the JSON response
    pick_df = pd.DataFrame(picks)
    pick_df.to_csv("debug_new_team.csv")
    
else:
    print(f"Error fetching team selection (Status Code: {response.status_code})")

for g in range(len(pick_df)):
    element=pick_df["element"].values[g]
    if(element in [109,536]):
        df=df[df["element_in"]!=element]
        element=304
        
    elif(element in [162]):
        df=df[df["element_in"]!=element]
        element=163
        
    if(element not in df["element_in"].values):
        new_row = pd.DataFrame({'element_in': [element], 'element_in_cost': [np.nan]}, index=[len(df)])
        df = pd.concat([df, new_row], ignore_index=True)

data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")
data=data[["Full_Name","element", "value", "kickoff_time"]]
data['kickoff_time'] = pd.to_datetime(data['kickoff_time'])
result = data.loc[data.groupby('Full_Name')['kickoff_time'].idxmax(), ['Full_Name','element', 'value', 'kickoff_time']]

team_df=pd.merge(df, result, left_on='element_in', right_on='element', how='left')

team_df['element_in_cost'] = team_df['element_in_cost'].fillna(team_df['value'])
team_df["selling_price_value"] = np.floor((team_df["value"] - team_df["element_in_cost"]) / 2).clip(lower=0)
team_df["selling_price"] = (team_df[["value", "element_in_cost"]].min(axis=1)+team_df["selling_price_value"])/10
pred_data=pd.read_csv("All_Predictions.csv").iloc[:,1:]["Name"]
team_df=team_df[team_df["element_in_cost"]>30]

name_map_dict={'Francisco Evanilson_de Lima Barbosa':'Francisco_Evanilson de Lima Barbosa'}
team_df['Full_Name'] =team_df['Full_Name'].replace(name_map_dict)

new_Names=[]
print(team_df)
name_list=pred_data.values
for j in range(len(team_df)):
    name=team_df["Full_Name"].values[j]
    if(name in name_list):
        new_Names.append(name)
    elif(name+'1' in name_list):
        new_Names.append(name+'1')
    elif(name+'0' in name_list):
        new_Names.append(name+'0')
print(new_Names)
team_df["Full_Name"]=new_Names     

team_df.to_csv("Squad_data.csv")


# Load Data
data = pd.read_csv("Optimize_players.csv")
data = data[~data["Name"].isin(banned_list)]
cols = ["p0","p1"]
for col in cols:
    data[col] = np.where(data["offset"] < 1, data[col] * data["offset"], data[col] * data["minutes_multiplier"])
    
squad=pd.read_csv("Squad_data.csv")

url = f"https://fantasy.premierleague.com/api/entry/{team_id}/"
response = requests.get(url)
if response.status_code == 200:
    resonsep_data = response.json()
    
else:
    print(f"Error fetching data (Status Code: {response.status_code})")

money_in_bank_init = resonsep_data.get("last_deadline_bank", 0)/10  # Convert to actual value
print("Money")
print(money_in_bank_init)

players = data['Name'].tolist()
costs = data['value'].tolist()
initial_squad=[]
for t in range (len(squad)):
    name=squad["Full_Name"].values[t]
    initial_squad.append(players.index(name))
    
"""initial_squad=[players.index('Ezri_Konsa Ngoyo'),players.index('Anthony_Gordon0'),players.index('Bukayo_Saka'),players.index('Ibrahima_Konaté'),
               players.index('Ismaïla_Sarr'),players.index('David_Raya Martin0'),players.index('Omar_Marmoush'),players.index('Alphonse_Areola'),
               players.index('Joško_Gvardiol'),players.index('Daniel_Muñoz'),players.index('Jurriën_Timber'),players.index('Jacob_Murphy'),
               players.index('Jean-Philippe_Mateta'),players.index('Alexander_Isak'),players.index('Mohamed_Salah')]"""

list1 = costs.copy()
selling_cost = squad["selling_price"].values
budget_amount=sum(selling_cost)+money_in_bank_init


# Update list1 with values from list2 at positions specified by indexes
for i in range(len(initial_squad)):
    list1[initial_squad[i]] = selling_cost[i]  

# Define Constants
#budget = 103.0  
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist()
#predicted_points = data[['p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8']].values
predicted_points = data[["p0","p1"]].values
# Initial Squad

# --- Define Gameweeks and Precompute Indices ---
optimize_range = 2 # Number of gameweeks to optimize
gameweeks = range(optimize_range)
num_players = len(players)

# Precompute position indices so we don’t iterate over all players each time.
def_indices   = [i for i, pos in enumerate(positions) if pos == 'DEF']
gk_indices    = [i for i, pos in enumerate(positions) if pos == 'GK']
mid_indices   = [i for i, pos in enumerate(positions) if pos == 'MID']
fwd_indices   = [i for i, pos in enumerate(positions) if pos == 'FWD']
outfield_indices = [i for i, pos in enumerate(positions) if pos != 'GK']

# Precompute team indices: dictionary mapping team to list of player indices
teams_set = set(teams)
team_to_indices = {team: [i for i, t in enumerate(teams) if t == team] for team in teams_set}

# --- Define Model and Decision Variables ---
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables
x            = {(i, t): LpVariable(f"x_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
bench        = {(i, t): LpVariable(f"bench_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
c            = {(i, t): LpVariable(f"captain_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
y            = {(i, t): LpVariable(f"y_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
transfer_in  = {(i, t): LpVariable(f"transfer_in_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
transfer_out = {(i, t): LpVariable(f"transfer_out_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
saved_transfers   = {t: LpVariable(f"saved_transfers_{t}", lowBound=0, upBound=5, cat='Integer') for t in gameweeks}
money_in_bank_var = {t: LpVariable(f"money_in_bank_{t}", lowBound=0, cat='Continuous') for t in gameweeks}

# --- Initial Squad Constraint (Gameweek 0) ---
for i in range(num_players):
    model += x[i, 0] == (1 if i in initial_squad else 0)

# --- Objective Function ---
# (Bench points term is added only if bench_points_gw is in the gameweek range)
obj = lpSum((y[i, t] + c[i, t]) * predicted_points[i][t] 
            for i in range(num_players) for t in gameweeks)
if bench_points_gw in gameweeks:
    obj += lpSum(bench[i, bench_points_gw] * predicted_points[i][bench_points_gw] 
                 for i in range(num_players))
model += obj

# --- Position Constraints ---
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players)) == 15
    model += lpSum(x[i, t] for i in def_indices) == 5
    model += lpSum(x[i, t] for i in gk_indices) == 2
    model += lpSum(x[i, t] for i in mid_indices) == 5
    model += lpSum(x[i, t] for i in fwd_indices) == 3

# --- Bench Constraints ---
for t in gameweeks:
    model += lpSum(bench[i, t] for i in gk_indices) == 1
    model += lpSum(bench[i, t] for i in outfield_indices) == 3
    for i in range(num_players):
        model += bench[i, t] <= x[i, t]

# --- Playing Status Constraints ---
for t in gameweeks:
    # Ensure at least 3 defenders are in the starting XI
    model += lpSum(y[i, t] for i in def_indices) >= 3
    for i in range(num_players):
        model += y[i, t] <= x[i, t]
        model += y[i, t] <= 1 - bench[i, t]
        model += y[i, t] >= x[i, t] - bench[i, t]

# --- Captain Selection ---
for t in gameweeks:
    model += lpSum(c[i, t] for i in range(num_players)) == 1
    for i in range(num_players):
        model += c[i, t] <= y[i, t]

# --- Budget Constraints ---
# Update money in bank for t >= 1
for t in gameweeks[1:]:
    model += money_in_bank_var[t] == money_in_bank_var[t-1] + \
             lpSum(transfer_out[i, t] * list1[i] for i in range(num_players)) - \
             lpSum(transfer_in[i, t] * costs[i] for i in range(num_players))
# For each gameweek, squad value (using list1) plus money in bank equals available funds.
for t in gameweeks:
    model += lpSum(x[i, t] * list1[i] for i in range(num_players)) + money_in_bank_var[t] == budget_amount

# --- Maximum 3 Players per Team ---
for t in gameweeks:
    for team, indices in team_to_indices.items():
        model += lpSum(x[i, t] for i in indices) <= 3

# --- Transfer Constraints ---
for t in gameweeks[1:]:
    if t == wildcard_round:
        # Wildcard round: unlimited transfers
        for i in range(num_players):
            model += x[i, t] >= x[i, t-1] - transfer_out[i, t]
            model += x[i, t] <= x[i, t-1] + transfer_in[i, t]
    else:
        # Normal transfer constraints
        for i in range(num_players):
            model += transfer_in[i, t] >= x[i, t] - x[i, t-1]
            model += transfer_out[i, t] >= x[i, t-1] - x[i, t]
            model += transfer_out[i, t] <= x[i, t-1]
        model += lpSum(transfer_in[i, t] for i in range(num_players)) <= 1 + saved_transfers[t-1]

for t in gameweeks[1:]:
    if t == wildcard_round:
        model += saved_transfers[t] == hit # Reset after wildcard
    else:
        model += saved_transfers[t] == saved_transfers[t-1] + (1 - lpSum(transfer_in[i, t] for i in range(num_players)))
        model += saved_transfers[t] <= 5

# --- Initial Transfers & Bank ---
model += saved_transfers[0] == initial_saved
model += money_in_bank_var[0] == money_in_bank_init

# --- Solve the Model ---
#model.solve(PULP_CBC_CMD(msg=True, timeLimit=400))
model.solve()
# Display Results
print(f"Status: {model.status}")
print(f"\nTotal Optimized Predicted Points: {model.objective.value()}")
# Display Squad & Transfers
for t in range(1, optimize_range):
    print(f"\nGameweek {t+Last_GW} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

# Display Transfers for Each Gameweek
for t in range(1, optimize_range):  
    print(f"\nTransfers for Gameweek {t+Last_GW}:")
    if t == wildcard_round:
        print("  Wildcard Activated! All players can be changed.")
    else:
        players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
        players_out = [players[i] for i in range(num_players) if x[i, t-1].varValue > 0.5 and x[i, t].varValue < 0.5]

        if players_in or players_out:
            print(f"  In: {', '.join(players_in) if players_in else 'None'}")
            print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
        else:
            print("  No transfers this week.")
# Display Captains
for t in range(1, optimize_range):
    for i in range(num_players):
        if c[i, t].varValue > 0.5:
            print(f"Gameweek {t+Last_GW} Captain: {players[i]}")

#252.26.6 m hit
#247.63 uten hit


#meg
#hit-64.95


<>:126: SyntaxWarning: invalid escape sequence '\F'
<>:126: SyntaxWarning: invalid escape sequence '\F'
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_46596\2551585297.py:126: SyntaxWarning: invalid escape sequence '\F'
  data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")


[{'name': 'wildcard', 'time': '2024-09-21T18:54:21.379105Z', 'event': 6}, {'name': '3xc', 'time': '2025-01-27T17:54:13.429445Z', 'event': 24}, {'name': 'manager', 'time': '2025-02-14T17:32:26.769885Z', 'event': 25}, {'name': 'wildcard', 'time': '2025-03-30T20:09:06.768246Z', 'event': 30}, {'name': 'bboost', 'time': '2025-04-18T21:31:47.116611Z', 'event': 33}, {'name': 'freehit', 'time': '2025-04-20T09:20:02.806131Z', 'event': 34}]
34
     element_in  element_in_cost  element_out  element_out_cost   entry  \
0            17              103           99                82  544468   
1           110               68          755                74  544468   
2           541               71          207                76  544468   
3            99               81           17               103  544468   
4           418               56          255                48  544468   
..          ...              ...          ...               ...     ...   
105          52               45     

In [ ]:
With hits

In [39]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import requests
import numpy as np

# -------------------------
# Settings and Team Info
# -------------------------
team_id = 544468
wildcard_round = 10      # Gameweek 3 (as defined)
bench_points_gw = 20
Last_GW = 29
hit_penalty = 1        # Penalty: -4 points for each extra transfer
hit = 0                 # Extra free transfer bonus (if any)

# -------------------------
# Get Transfer Data from FPL API
# -------------------------
def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{team_id}/transfers/"
    response_transfers = requests.get(transfers_url)
    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None
    return response_transfers.json()

team_transfers = get_transfers(team_id)
df = pd.DataFrame(team_transfers)

#Find if free hit used
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/history/"
resp = requests.get(url)
if resp.status_code == 200:
    chip_data = resp.json()
    chips = chip_data.get("chips", [])
    print(chips)
    # find the Free Hit chip
    freehit = next((c for c in chips if c.get("name") == "freehit"), None)
    if freehit:
        free_hit_gw_played = freehit.get("event")
    else:
        free_hit_gw_played=39

        
# Compute number of transfers per gameweek
transfers1 = df.groupby('event').size().reset_index(name='count')
max_event = Last_GW
print("Max event:", max_event)

# Calculate saved transfers over past gameweeks
saved_transfers = 0
last_event = 0
for h in range(max_event):
    new_event = last_event + 1
    if new_event in transfers1["event"].values:
        ind = transfers1["event"].tolist().index(new_event)
        transfers_made = transfers1["count"].values[ind]
        saved_transfers = saved_transfers - transfers_made
        saved_transfers = max(0, saved_transfers)
    else:
        saved_transfers += 1
    last_event = new_event
initial_saved = saved_transfers + hit
print("Initial saved transfers:", initial_saved)

# -------------------------
# Mark Active Transfers and Process Picks
# -------------------------
active = []
for i in range(len(df["element_in"])):
    element_in = df["element_in"].values[-i-1]
    out_list = df["element_out"].iloc[0:-i-1].values
    if element_in in out_list:
        active.append(0)
    else:
        active.append(1)
df["Active"] = list(reversed(active))
df = df[df["Active"] == 1]
df = df[["element_in", "element_in_cost"]]

# Get team picks for the current gameweek (Last_GW)
gameweek = Last_GW
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/event/{gameweek}/picks/"
response = requests.get(url)
if response.status_code == 200:
    team_selection = response.json()
    picks = team_selection.get("picks")
    pick_df = pd.DataFrame(picks)
else:
    print(f"Error fetching team selection (Status Code: {response.status_code})")

# Process picks – adjust specific elements as needed
for g in range(len(pick_df)):
    element = pick_df["element"].values[g]
    if element in [109, 536]:
        df = df[df["element_in"] != element]
        element = 304
    elif element in [162]:
        df = df[df["element_in"] != element]
        element = 163
    if element not in df["element_in"].values:
        new_row = pd.DataFrame({'element_in': [element], 'element_in_cost': [np.nan]}, index=[len(df)])
        df = pd.concat([df, new_row], ignore_index=True)

# -------------------------
# Merge with Raw Data and Adjust Squad Data
# -------------------------
data = pd.read_csv("Raw_Data_24\\Fantasy_season_2024_data.csv")
data = data[["Full_Name", "element", "value", "kickoff_time"]]
data['kickoff_time'] = pd.to_datetime(data['kickoff_time'])
result = data.loc[data.groupby('Full_Name')['kickoff_time'].idxmax(), 
                    ['Full_Name', 'element', 'value', 'kickoff_time']]

team_df = pd.merge(df, result, left_on='element_in', right_on='element', how='left')
team_df['element_in_cost'] = team_df['element_in_cost'].fillna(team_df['value'])
team_df["selling_price_value"] = np.floor((team_df["value"] - team_df["element_in_cost"]) / 2).clip(lower=0)
team_df["selling_price"] = (team_df[["value", "element_in_cost"]].min(axis=1) + team_df["selling_price_value"]) / 10
pred_data = pd.read_csv("All_Predictions.csv").iloc[:, 1:]["Name"]
team_df = team_df[team_df["element_in_cost"] > 30]
new_Names = []
name_list = pred_data.values
for j in range(len(team_df)):
    name = team_df["Full_Name"].values[j]
    if name in name_list:
        new_Names.append(name)
    elif name + '1' in name_list:
        new_Names.append(name + '1')
    elif name + '0' in name_list:
        new_Names.append(name + '0')
team_df["Full_Name"] = new_Names     
team_df.to_csv("Squad_data.csv", index=False)

# -------------------------
# Load Optimization Data
# -------------------------
data = pd.read_csv("Optimize_players.csv")
squad = pd.read_csv("Squad_data.csv")

# Get bank value from FPL API
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/"
response = requests.get(url)
if response.status_code == 200:
    resonsep_data = response.json()
else:
    print(f"Error fetching data (Status Code: {response.status_code})")
money_in_bank_init = resonsep_data.get("last_deadline_bank", 0) / 10

players = data['Name'].tolist()
costs = data['value'].tolist()
initial_squad = []
for t in range(len(squad)):
    name = squad["Full_Name"].values[t]
    initial_squad.append(players.index(name))

# Create a cost list where players already in the squad use their selling price
list1 = costs.copy()
selling_cost = squad["selling_price"].values
budget_amount = sum(selling_cost) + money_in_bank_init
print("Budget amount:", budget_amount)
for i in range(len(selling_cost)):
    list1[initial_squad[i]] = selling_cost[i]

# Other constants from Optimize_players.csv
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist()
predicted_points = data[['p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8']].values

# -------------------------
# Build the LP Optimization Model
# -------------------------
# Define gameweeks (e.g. 9 gameweeks to optimize)
optimize_range = 4
gameweeks = range(optimize_range)
num_players = len(players)

# Precompute position indices for efficiency
def_indices     = [i for i, pos in enumerate(positions) if pos == 'DEF']
gk_indices      = [i for i, pos in enumerate(positions) if pos == 'GK']
mid_indices     = [i for i, pos in enumerate(positions) if pos == 'MID']
fwd_indices     = [i for i, pos in enumerate(positions) if pos == 'FWD']
outfield_indices = [i for i, pos in enumerate(positions) if pos != 'GK']

# Precompute team indices: mapping team -> list of player indices
teams_set = set(teams)
team_to_indices = {team: [i for i, t in enumerate(teams) if t == team] for team in teams_set}

# Create the LP model
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables
x            = {(i, t): LpVariable(f"x_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
bench        = {(i, t): LpVariable(f"bench_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
c            = {(i, t): LpVariable(f"captain_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
y            = {(i, t): LpVariable(f"y_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
transfer_in  = {(i, t): LpVariable(f"transfer_in_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
transfer_out = {(i, t): LpVariable(f"transfer_out_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
saved_transfers = {t: LpVariable(f"saved_transfers_{t}", lowBound=0, upBound=5, cat='Integer') for t in gameweeks}
money_in_bank_var = {t: LpVariable(f"money_in_bank_{t}", lowBound=0, cat='Continuous') for t in gameweeks}

# Extra transfers variable (for t > 0) to capture transfers beyond free allowance
extra_transfers = {t: LpVariable(f"extra_transfers_{t}", lowBound=0, cat='Integer') for t in gameweeks if t > 0}

# --- Initial Squad Constraint (Gameweek 0) ---
for i in range(num_players):
    model += x[i, 0] == (1 if i in initial_squad else 0)

# --- Objective Function ---
# Sum predicted points for starting XI and captain bonus.
obj = lpSum((y[i, t] + c[i, t]) * predicted_points[i][t] for i in range(num_players) for t in gameweeks)
if bench_points_gw in gameweeks:
    obj += lpSum(bench[i, bench_points_gw] * predicted_points[i][bench_points_gw] for i in range(num_players))
# Subtract hit penalty for extra transfers (only in normal gameweeks: t>0 and t != wildcard_round)
for t in gameweeks:
    if t > 0 and t != wildcard_round:
         obj -= hit_penalty * extra_transfers[t]
model += obj

# --- Position Constraints ---
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players)) == 15
    model += lpSum(x[i, t] for i in def_indices) == 5
    model += lpSum(x[i, t] for i in gk_indices) == 2
    model += lpSum(x[i, t] for i in mid_indices) == 5
    model += lpSum(x[i, t] for i in fwd_indices) == 3

# --- Bench Constraints ---
for t in gameweeks:
    model += lpSum(bench[i, t] for i in gk_indices) == 1
    model += lpSum(bench[i, t] for i in outfield_indices) == 3
    for i in range(num_players):
        model += bench[i, t] <= x[i, t]

# --- Playing Status Constraints ---
for t in gameweeks:
    model += lpSum(y[i, t] for i in def_indices) >= 3
    for i in range(num_players):
        model += y[i, t] <= x[i, t]
        model += y[i, t] <= 1 - bench[i, t]
        model += y[i, t] >= x[i, t] - bench[i, t]

# --- Captain Selection ---
for t in gameweeks:
    model += lpSum(c[i, t] for i in range(num_players)) == 1
    for i in range(num_players):
        model += c[i, t] <= y[i, t]

# --- Budget Constraints ---
for t in gameweeks[1:]:
    model += money_in_bank_var[t] == money_in_bank_var[t-1] + \
             lpSum(transfer_out[i, t] * list1[i] for i in range(num_players)) - \
             lpSum(transfer_in[i, t] * costs[i] for i in range(num_players))
for t in gameweeks:
    model += lpSum(x[i, t] * list1[i] for i in range(num_players)) + money_in_bank_var[t] == budget_amount

    
# --- Maximum 3 Players per Team ---
for t in gameweeks:
    for team, indices in team_to_indices.items():
        model += lpSum(x[i, t] for i in indices) <= 3

# --- Transfer Constraints ---
for t in gameweeks[1:]:
    if t == wildcard_round:
        # Wildcard round: unlimited transfers; no hit penalty.
        for i in range(num_players):
            model += x[i, t] >= x[i, t-1] - transfer_out[i, t]
            model += x[i, t] <= x[i, t-1] + transfer_in[i, t]
        if t in extra_transfers:
            model += extra_transfers[t] == 0
    else:
        # Normal gameweeks:
        for i in range(num_players):
            model += transfer_in[i, t] >= x[i, t] - x[i, t-1]
            model += transfer_out[i, t] >= x[i, t-1] - x[i, t]
            model += transfer_out[i, t] <= x[i, t-1]
        free_allowance = 1 + saved_transfers[t-1]
        model += lpSum(transfer_in[i, t] for i in range(num_players)) <= saved_transfers[t] + extra_transfers[t]
        model += extra_transfers[t] >= lpSum(transfer_in[i, t] for i in range(num_players)) - saved_transfers[t]
        model += extra_transfers[t] >= 0

# --- Saved Transfers Updates ---
for t in gameweeks[1:]:
    if t == wildcard_round:
        model += saved_transfers[t] == 0
    else:
        # If transfers exceed free allowance, we set saved transfers to 0.
        # Otherwise, saved transfers carry over as: saved_transfers[t-1] + 1 - transfers_used.
        model += saved_transfers[t] <= saved_transfers[t-1] + 1 - lpSum(transfer_in[i, t] for i in range(num_players))
        # (The lower bound of saved_transfers is already 0.)

# --- Initial Transfers & Bank ---
model += saved_transfers[0] == initial_saved
model += money_in_bank_var[0] == money_in_bank_init

# -------------------------
# Solve the Model
# -------------------------
solver = PULP_CBC_CMD(msg=True, timeLimit=550)
model.solve()

# -------------------------
# Display Results
# -------------------------
print(f"Status: {model.status}")
print(f"Total Optimized Predicted Points: {model.objective.value()}")

for t in range(1, optimize_range):
    print(f"\nGameweek {t+Last_GW} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

for t in range(1, optimize_range):
    print(f"\nTransfers for Gameweek {t+Last_GW}:")
    if t == wildcard_round:
        print("  Wildcard Activated! All players can be changed.")
    else:
        players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
        players_out = [players[i] for i in range(num_players) if x[i, t-1].varValue > 0.5 and x[i, t].varValue < 0.5]
        if players_in or players_out:
            print(f"  In: {', '.join(players_in) if players_in else 'None'}")
            print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
        else:
            print("  No transfers this week.")

for t in range(1, optimize_range):
    for i in range(num_players):
        if c[i, t].varValue > 0.5:
            print(f"Gameweek {t+Last_GW} Captain: {players[i]}")


[{'name': 'wildcard', 'time': '2024-09-21T18:54:21.379105Z', 'event': 6}, {'name': '3xc', 'time': '2025-01-27T17:54:13.429445Z', 'event': 24}, {'name': 'manager', 'time': '2025-02-14T17:32:26.769885Z', 'event': 25}, {'name': 'wildcard', 'time': '2025-03-30T20:09:06.768246Z', 'event': 30}, {'name': 'bboost', 'time': '2025-04-18T21:31:47.116611Z', 'event': 33}, {'name': 'freehit', 'time': '2025-04-20T09:20:02.806131Z', 'event': 34}]
Max event: 29
Initial saved transfers: 0
Budget amount: 191.00000000000003


KeyError: "['p4', 'p5', 'p6', 'p7', 'p8'] not in index"

# 

In [37]:
import requests

team_id = 6  # your team ID here
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/history/"

resp = requests.get(url)
if resp.status_code == 200:
    data = resp.json()
    chips = data.get("chips", [])
    print(chips)
    
    # find the Free Hit chip
    freehit = next((c for c in chips if c.get("name") == "freehit"), None)
    if freehit:
        free_hit_gw_played = freehit.get("event")
    else:
        free_hit_gw_played=39


[{'name': '3xc', 'time': '2024-08-24T08:00:20.768432Z', 'event': 2}, {'name': 'wildcard', 'time': '2024-08-28T00:45:22.448973Z', 'event': 3}, {'name': 'manager', 'time': '2025-01-31T20:45:10.872739Z', 'event': 24}, {'name': 'freehit', 'time': '2025-03-11T00:30:33.023154Z', 'event': 29}, {'name': 'wildcard', 'time': '2025-04-01T17:11:16.637624Z', 'event': 30}, {'name': 'bboost', 'time': '2025-04-12T09:55:30.938742Z', 'event': 32}]
Free Hit was used in Gameweek 29
